# BUSCA DOS PARÂMETROS DO GERADOR SINTÉTICO

Otimização dos 23 parâmetros do `Synthetic/index.py` com a **DE auto-adaptativa** (`Models/AdaptiveDE`) do
framework `Nature`, maximizando o **IoU no `dataset_wu`** de um modelo treinado só com dado sintético.
A aptidão é o pipeline inteiro do projeto, rodado por papermill exatamente como o `Task/index.py` faz:

```
options do DE → Synthetic.dataset(220)             →  Dataset/dataset_trial/original/
              → Dataset/dataset_trial/Format.ipynb →  images/, masks/, Dataset/DataBase.csv
              → Model/Analysis.ipynb               →  Model/Backup/model_<n>/
              → Model/Predict.ipynb (dataset_wu)   →  iou_wu  ← objetivo
```

Quanto maior o IoU, melhor o sintético reproduz o que importa do `dataset_wu` — é a definição operacional de
"clone" aqui: um dado que ensina ao modelo o que o `wu` ensinaria.

**Nenhum arquivo do projeto é modificado.** Os notebooks são lidos, corrigidos em memória (só o `modelId` e o
`dataset` do `Predict`) e executados a partir de `files/runs/` com o `cwd` da pasta original — a mesma
mecânica do `Task/index.py`. O que a busca escreve é o que o pipeline normal já escreveria: `Task/info.json`,
`Dataset/dataset_trial/`, `Dataset/DataBase.csv` e um `Model/Backup/model_<n>` por avaliação.

### CUSTO — MEDIDO NESTA MÁQUINA
| etapa | custo | medição |
|---|---|---|
| gerar 220 volumes | **≈9 min** | 42,5 s por volume, 20 processos |
| `Format` (percentil global + 220 `.npy`) | ≈3 min | |
| **treino** | **3 a 8 h** | 2,9 s/step → ≈4,8 min/época com 200 volumes; 100 épocas com early stopping |
| `Predict` nos 220 volumes do `wu` | ≈3 min | |

Uma avaliação custa portanto **4 a 9 horas**, e `POPULATION · GENERATIONS = 1500` avaliações seriam ~10 meses
de máquina. Três alavancas, em ordem de impacto e todas sem tocar em nenhum arquivo do projeto:

* **`TRAIN_EPOCHS`** — teto de épocas do trial (o `Model/Analysis.ipynb` usa 100 fixo). Em 25 o treino cai
  para ~2 h. Como todas as configurações competem com o mesmo orçamento, a comparação continua justa; o que
  muda é o IoU absoluto, não a ordem entre elas.
* **`N_IMAGES`** — o tempo de época é proporcional. 60 volumes deixam a época em ~1,5 min.
* **`TRIAL_INFO['network']`** — `unet3d_v2` com `num_filters=16` treina muito mais rápido que o `resaceunet`
  de 32. Vale se o objetivo é ranquear geradores, não produzir o modelo final.

Qualquer um dos três entra na chave do cache, então mudar de ideia no meio não mistura medições
incompatíveis — só custa reavaliar.

### RETOMADA
Duas camadas de persistência tornam a campanha interrompível a qualquer momento:

* **`files/optimization/`** — o `memory=` do `Nature`: população, RNG e histórico. O `state.npz` só é gravado
  a cada `Memory.EVERY = 10` gerações, e o `best.json`/`history.json` no fim de cada campanha.
* **`files/trials.json`** — cache próprio, gravado **a cada avaliação**. É o que cobre o intervalo entre os
  saves da `Memory`: o `Randomizer` do `Nature` é determinístico (`SEED` fixo), então retomar reexecuta a
  mesma sequência de genomas e tudo que já foi avaliado volta do cache em milissegundos. É também a tabela
  de análise da campanha.

Interromper pelo kernel (⏹) é seguro: sem `commit()` a campanha não fecha, e a próxima chamada de `update()`
**continua o ciclo original**, no cronograma original.

### ALTERAR `population` COM A MEMÓRIA ATIVA
Pode. `Memory.start()` só rejeita um estado salvo quando muda a **variante** (`lshade`) ou a **lista de
variáveis** (`names`); `population` e `generations` não entram nessa validação. O efeito de cada um:

* **`population`** — numa *extensão* (campanha anterior fechada por `commit()`), se a nova população for maior
  que a salva o `AdaptiveDE` **reinfla** com indivíduos novos e mantém os sobreviventes como elite. Numa
  *retomada* (campanha interrompida no meio) a população salva segue como está.
* **`generations`** — junto com `population` define `maxNfe = population · generations`, que é o horizonte do
  ciclo **e** a curva do LPSR (a redução linear da população). Mudar qualquer um dos dois altera o *ritmo*
  daqui para a frente, nunca o que já foi aprendido.
* **`SEARCH_SPACE`** — mudar as variáveis (nomes, ordem ou quantidade) **invalida** a pasta: a `Memory`
  levanta `ValueError` dizendo que ela pertence a outro problema. Aí é só apontar `MEMORY_DIR` para uma pasta
  nova; o `files/trials.json` continua valendo, porque é indexado pelas `options` do gerador e não pelo genoma.

In [ ]:
import os, re, sys, json, shutil, hashlib, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nbformat
import papermill as pm
from scipy import ndimage
from pathlib import Path
from datetime import datetime
from threading import Thread
from tqdm.auto import tqdm
from time import time, sleep

sys.path.insert(0, str(Path('Nature').resolve()))
from Models.AdaptiveDE.index import AdaptiveDE   # direto: o NatureSelector importa o Genetic, que exige deap

pd.set_option('display.max_columns', None)
print(sys.executable)

# CONSTANTES
Caminhos do pipeline e orçamento da campanha. `TRIAL_INFO` sai do `Task/info.json` que já está no projeto —
a busca não escolhe rede nem hiperparâmetro de treino, só força o dataset do trial e o modo sem tiles.

In [ ]:
ROOT      = Path('..').resolve()
SYNTHETIC = ROOT / 'Synthetic'
TRIAL_DIR = ROOT / 'Dataset' / 'dataset_trial'
MODEL_DIR = ROOT / 'Model'
BACKUP    = MODEL_DIR / 'Backup'
DATABASE  = ROOT / 'Dataset' / 'DataBase.csv'
INFO_PATH = ROOT / 'Task' / 'info.json'

FILES       = Path('files').resolve()
RUNS_DIR    = FILES / 'runs'
MEMORY_DIR  = FILES / 'optimization'
TRIALS_PATH = FILES / 'trials.json'
os.makedirs(RUNS_DIR, exist_ok=True)

KERNEL_NAME     = 'python3'
PREDICT_DATASET = 'dataset_wu'
N_IMAGES     = 220
N_JOBS       = os.cpu_count()
DATA_SEED    = 7
FAILED_SCORE = 0.0
MIN_FAULT    = 0.002   # 0,2% dos voxels; o dataset_wu tem 6,9% e nunca menos de 3,6%
MIN_SPREAD   = 0.1     # o gerador devolve z-score, então std ≈ 1; perto de 0 o volume saturou num valor só
TRAIN_EPOCHS = None   # None mantém as 100 épocas do Model/Analysis.ipynb; um inteiro reduz o custo do trial

POPULATION  = 15
GENERATIONS = 100
VARIANT     = 'lshade'
PATIENCE    = None
SEED        = 42

KEEP_WEIGHTS = False   # apaga só o model.pth do trial; info.json, train.png e predictions/ ficam
KEEP_LOGS    = False   # guarda os notebooks executados apenas quando o trial falha

TRIAL_INFO = {**json.loads(INFO_PATH.read_text()), 'dataset': TRIAL_DIR.name, 'img_size': None}
TRIAL_INFO

# ESPAÇO DE BUSCA
Os 23 parâmetros do `SyntheticGenerator`. Os 14 que são **intervalos** viram duas variáveis (`_lo`/`_hi`) e os
9 escalares entram direto — **37 variáveis**. Os limites partem dos defaults da classe e do
`Dataset/dataset_trial/synthetic.json`, alargados para dar espaço à busca.

Quatro limites não são o intervalo "natural" do parâmetro, e sim o intervalo **útil**, medido (a seção
seguinte mostra as medições). Alargá-los não amplia a busca — só gasta avaliações de horas em regiões onde o
parâmetro perdeu o efeito:

* `layerRange`, `layerThickness`, `foldCount` e `faultCount` alimentam `np.random.randint(lo, hi)`, que exige
  `hi > lo` — o `toOptions()` ordena o par e abre `hi = lo + 1` quando os dois caem no mesmo inteiro.
* **`faultCount` começa em 1, não em 0**: com o par colapsado em `[0, 1]` o `randint` devolve 0 em *todos* os
  tiles e o dataset inteiro sai sem falhas — 220 volumes e um treino de horas para um IoU garantidamente zero.
* **`waveletDt` ∈ [0.0010, 0.0018]**, não até 0.004. O pico do espectro é exatamente `waveletFreq · waveletDt`
  (verificado), então com `dt ≥ 0.003` e frequência alta o pico passa de Nyquist (0.5) e a Ricker encolhe para
  3 amostras — vira um impulso e a convolução deixa de fazer efeito. Com este limite, `waveletFreq ∈ [20, 160]`
  varre o pico de **0.020 a 0.288**, e o alvo do `wu` (0.1156) cai no meio da faixa.
* **`faultZoneWidth` começa em 0.6**: a máscara é discretizada em voxels e tem piso de ~2 voxels de espessura,
  então abaixo disso o parâmetro não muda mais nada (0.4 e 0.99 produzem a mesma espessura medida).

In [ ]:
SEARCH_SPACE = {
    'layerRange':      {'type': 'int',  'pair': True,  'bounds': (10, 300)},
    'layerThickness':  {'type': 'int',  'pair': True,  'bounds': (1, 8)},
    'foldCount':       {'type': 'int',  'pair': True,  'bounds': (0, 60)},
    'foldSigma':       {'type': 'real', 'pair': True,  'bounds': (1.0, 90.0)},
    'foldAmplitude':   {'type': 'real', 'pair': True,  'bounds': (-60.0, 60.0)},
    'foldBaseShift':   {'type': 'real', 'pair': True,  'bounds': (-10.0, 10.0)},
    'foldDamping':     {'type': 'real', 'pair': False, 'bounds': (0.2, 3.0)},
    'shearOffset':     {'type': 'real', 'pair': True,  'bounds': (-15.0, 15.0)},
    'shearGradient':   {'type': 'real', 'pair': True,  'bounds': (-0.3, 0.3)},
    'faultCount':      {'type': 'int',  'pair': True,  'bounds': (1, 14)},
    'faultThrow':      {'type': 'real', 'pair': True,  'bounds': (0.0, 50.0)},
    'faultDipAngle':   {'type': 'real', 'pair': True,  'bounds': (5.0, 89.0)},
    'faultDecaySigma': {'type': 'real', 'pair': True,  'bounds': (5.0, 150.0)},
    'faultRoughness':  {'type': 'real', 'pair': False, 'bounds': (0.0, 10.0)},
    'faultRoughSigma': {'type': 'real', 'pair': False, 'bounds': (1.0, 20.0)},
    'faultZoneWidth':  {'type': 'real', 'pair': False, 'bounds': (0.6, 4.0)},
    'faultThreshold':  {'type': 'real', 'pair': False, 'bounds': (0.05, 5.0)},
    'faultCurveProb':  {'type': 'real', 'pair': False, 'bounds': (0.0, 1.0)},
    'faultCurveMax':   {'type': 'real', 'pair': False, 'bounds': (0.0, 20.0)},
    'waveletFreq':     {'type': 'real', 'pair': True,  'bounds': (20.0, 160.0)},
    'waveletDuration': {'type': 'real', 'pair': False, 'bounds': (0.04, 0.12)},
    'waveletDt':       {'type': 'real', 'pair': False, 'bounds': (0.0010, 0.0018)},
    'noiseLevel':      {'type': 'real', 'pair': True,  'bounds': (0.0, 1.0)},
}

VARIABLES = {}

for name, spec in SEARCH_SPACE.items():
    gene = {'type': spec['type'], 'bounds': spec['bounds']}

    if not spec['pair']:
        VARIABLES[name] = gene
        continue

    VARIABLES[f'{name}_lo'] = dict(gene)
    VARIABLES[f'{name}_hi'] = dict(gene)

print(f'{len(SEARCH_SPACE)} parâmetros -> {len(VARIABLES)} variáveis')
pd.DataFrame([{'variavel': k, 'tipo': v['type'], 'min': v['bounds'][0], 'max': v['bounds'][1]} for k, v in VARIABLES.items()])

### DO GENOMA PARA O GERADOR
`toOptions()` traduz o dict decodificado pelo `Problem` no dict que o `SyntheticGenerator.set()` espera, e
`toValues()` faz o caminho de volta (é o que permite passar um preset existente pela função objetivo).
Ordenar o par em vez de restringi-lo é de propósito: uma `constraint` gastaria avaliações de horas só para
rejeitar genomas invertidos que são perfeitamente utilizáveis ao contrário.

In [ ]:
def toOptions(values):
    options = {}

    for name, spec in SEARCH_SPACE.items():
        if not spec['pair']:
            options[name] = float(values[name])
            continue

        lo, hi = sorted([values[f'{name}_lo'], values[f'{name}_hi']])

        if spec['type'] == 'int':
            lo, hi = int(lo), max(int(hi), int(lo) + 1)
            options[name] = [lo, hi]
            continue

        options[name] = [float(lo), float(hi)]

    return options


def toValues(options):
    values = {}

    for name, spec in SEARCH_SPACE.items():
        if not spec['pair']:
            values[name] = float(options[name])
            continue

        values[f'{name}_lo'], values[f'{name}_hi'] = options[name]

    return values


BASELINE        = json.loads((TRIAL_DIR / 'synthetic.json').read_text())
BASELINE_VALUES = toValues(BASELINE)
outside = [k for k, v in BASELINE_VALUES.items() if not VARIABLES[k]['bounds'][0] <= v <= VARIABLES[k]['bounds'][1]]

print('preset fora dos limites:', outside or 'nenhum')
pd.DataFrame({'baseline (synthetic.json)': BASELINE, 'centro do espaço de busca': toOptions({k: np.mean(v['bounds']) for k, v in VARIABLES.items()})})

# O ALVO: O QUE O `dataset_wu` EXIGE DO GERADOR
Um espaço de busca só é "abrangente" se a estatística do alvo estiver **dentro** do que o gerador consegue
produzir. A célula abaixo mede o `dataset_wu` e confronta com o envelope do `SyntheticGenerator`, levantado
gerando volumes nos extremos de cada limite (14 configurações, uma por canto relevante do espaço).

| métrica | `dataset_wu` | alcance do gerador | veredito |
|---|---|---|---|
| falha (% dos voxels) | **6,94** (3,6 – 8,9) | 0 – 46,5 | coberto, alvo no terço inferior |
| planos de falha por volume | **2,9** (1 – 6) | 1 – 3 | coberto só com `faultCount` baixo |
| mergulho do plano | **76,5°** (p10 68, p90 86) | 8° – 79° | coberto na borda superior |
| pico do espectro vertical | **0,1156** ciclos/amostra | 0,020 – 0,288 | coberto, alvo no meio |
| espessura da falha | **2,11** voxels | 2,0 – 4,7 | coberto no piso |
| desvio-padrão pós-`Format` | **0,193** | 0,18 – 0,22 | coberto |

Três leituras que valem mais que a tabela:

1. **O preset atual está longe do alvo em duas frentes**: gera o espectro em 0,070 (alvo 0,1156 — dado mais
   "grosso" que o `wu`) e usa `faultCount = [5, 10]`, que funde as falhas num único plano conexo enquanto o
   `wu` tem ~3 separados. São exatamente as direções em que a busca tem espaço para ganhar.
2. **O eixo temporal do `wu` é o 1**, e é onde o gerador põe a wavelet depois do `transpose(0, 2, 1)` do
   `_generate_single`. Convenção confirmada — o espectro do `wu` bate no eixo certo.
3. **A normalização p01/p99 do `Format` apaga o efeito do `noiseLevel` sobre o desvio-padrão** (ruído zero e
   ruído máximo dão o mesmo 0,21). O parâmetro continua mudando a textura, não a escala — não espere que ele
   apareça na estatística global.
4. **O espaço é enviesado para datasets com mais falha que o alvo.** Sorteando 16 configurações uniformes
   dentro dos limites, a fração de falha saiu entre 6,4% e 43,7% — o `wu`, com 6,9%, fica na borda inferior.
   A busca vai ter que empurrar `faultCount`/`faultZoneWidth` para baixo, e é mais uma razão para não gastar
   avaliação com região morta.

### O QUE O ESPAÇO NÃO CONSEGUE PRODUZIR
As mesmas 16 configurações aleatórias mais os dois cantos extremos (tudo no mínimo, tudo no máximo) foram
geradas para valer: **nenhuma quebrou o gerador e nenhuma produziu valor não-finito**. Os dois cantos, porém,
são degenerados de um jeito que só aparece horas depois, no fim do treino:

* **canto mínimo** → máscara vazia (0% de falha);
* **canto máximo** → volume saturado num valor só (`std` = 0); aí o `Format` calcula `p01 == p99` e divide por
  zero, e o dataset inteiro vira `NaN`. O treino roda, a rede prediz vazio e o IoU sai 0.

Nenhum dos dois é alcançável por acidente no meio do espaço, mas ao longo de centenas de avaliações a DE
visita bordas. Por isso o `Trial` mede a fração de falha e o contraste **logo depois de gerar** e aborta em
segundos (`MIN_FAULT`, `MIN_SPREAD`) em vez de pagar o treino para chegar em zero.

In [ ]:
ENVELOPE = {   # medido gerando volumes nos extremos de cada limite do SEARCH_SPACE
    'falha %':        (0.0, 46.5),
    'planos':         (1.0, 3.0),
    'mergulho':       (8.3, 79.4),
    'pico espectral': (0.020, 0.288),
    'espessura':      (2.00, 4.72),
    'std':            (0.180, 0.219),
}


def measure(imgDir, mskDir, n=20):
    images = sorted(Path(imgDir).glob('*.npy'))[:n]
    masks  = sorted(Path(mskDir).glob('*.npy'))[:n]
    frac, planes, dips, peaks, thick, stds = [], [], [], [], [], []

    for imgPath, mskPath in zip(images, masks):
        img, msk = np.load(imgPath).astype(np.float64), np.load(mskPath) > 0
        stds.append(img.std())
        frac.append(100 * msk.mean())

        traces = np.moveaxis(img - img.mean(), 1, -1).reshape(-1, img.shape[1])[::64]
        peaks.append(np.fft.rfftfreq(img.shape[1])[np.argmax(np.abs(np.fft.rfft(traces, axis=-1)).mean(0))])

        if not msk.any():
            continue

        thick.append(2 * ndimage.distance_transform_edt(msk)[msk].mean())
        labels, total = ndimage.label(msk)
        big = 0

        for i in range(1, total + 1):
            points = np.argwhere(labels == i)
            if len(points) < 500:
                continue
            big += 1
            values, vectors = np.linalg.eigh(np.cov((points - points.mean(0)).T))
            dips.append(np.degrees(np.arccos(abs(vectors[:, 0][1]))))

        planes.append(big)

    return {'falha %': np.mean(frac), 'planos': np.mean(planes), 'mergulho': np.mean(dips),
            'pico espectral': np.mean(peaks), 'espessura': np.mean(thick), 'std': np.mean(stds)}


target = measure(ROOT / 'Dataset' / PREDICT_DATASET / 'images', ROOT / 'Dataset' / PREDICT_DATASET / 'masks')
df = pd.DataFrame({'alvo': target, 'gerador min': {k: v[0] for k, v in ENVELOPE.items()},
                   'gerador max': {k: v[1] for k, v in ENVELOPE.items()}})
df['coberto'] = [ENVELOPE[k][0] <= target[k] <= ENVELOPE[k][1] for k in df.index]
df

# ACOMPANHAMENTO DAS ETAPAS
As quatro etapas de uma avaliação rodam **fora deste notebook** — a geração num subprocesso, as outras três
num kernel do papermill. Cada uma tem o próprio `tqdm` lá dentro (o `dataset()` do gerador, o `Computing IoU`
do `Predict`), mas nada disso chega até aqui: o subprocesso tem a saída capturada e o papermill guarda a do
kernel filho no `.ipynb` de saída.

`Progress` resolve por fora: roda a etapa numa thread e desenha a barra a partir do que ela **vai deixando em
disco** — os `.npy` que aparecem na pasta, a época que o `Model/progress.json` registra. Sem contador, a barra
vira um cronômetro, que já mostra que a coisa está andando.

As barras usam `leave=False` de propósito: com 4 por avaliação e centenas de avaliações, deixá-las na tela
acumularia milhares de linhas e travaria o navegador. Some cada uma ao terminar, e o que fica é a barra da
campanha (a do próprio `AdaptiveDE`) mais a etapa em curso.

Duas coisas normais de ver e que não são travamento:

* **a barra de geração anda em degraus** — os volumes saem em `N_JOBS` processos paralelos e chegam ao disco
  em lotes, então ela salta de ~20 em ~20 a cada ~42 s, em vez de subir de um em um;
* **a barra de treino costuma terminar antes do fim** — quem encerra é o early stopping do
  `Model/Analysis.ipynb`, não o teto de épocas.

In [ ]:
class Progress:
    STEP = 2

    def __init__(self, desc, total=None, count=None, unit='it'):
        self.desc  = desc
        self.total = total
        self.count = count
        self.unit  = unit

    def update(self, action):
        result = {}

        def target():
            try:
                result['value'] = action()
            except BaseException as error:
                result['error'] = error

        worker = Thread(target=target, daemon=True)
        worker.start()

        with tqdm(total=self.total, desc=self.desc, unit=self.unit, leave=False) as bar:
            while worker.is_alive():
                worker.join(self.STEP)

                if self.count is None:
                    bar.update(self.STEP)   # sem contador em disco: a barra mede o tempo decorrido
                    continue

                bar.n = min(self.count(), self.total)
                bar.refresh()

        if 'error' in result:
            raise result['error']

        return result.get('value')


Progress('exemplo', 6, unit='s').update(lambda: sleep(4))
print('Progress ok')

# EXECUÇÃO DE NOTEBOOKS
`NotebookRunner` é o `Task/index.py` com duas capacidades a mais, necessárias porque nenhum arquivo fora
desta pasta pode ser tocado:

* **`replace`** — regex linha a linha antes de executar. Serve para o `Model/Predict.ipynb`, que tem o
  `modelId` e o `dataset` escritos na mão, e para o teto de épocas do `Model/Analysis.ipynb` quando
  `TRAIN_EPOCHS` está definido. As cópias corrigidas vão para `files/runs/`; os originais ficam intactos.
* **`drop`** — remove células por trecho do código. Usado para cortar as 15 figuras de predição do `Predict`,
  que custariam 15 inferências extras por avaliação.

`tolerate` existe por causa do `Format.ipynb`: no modo sem tiles ele termina em `sys.exit()`, que o papermill
converte em `PapermillExecutionError` **depois** de o `DataBase.csv` já ter sido escrito. Quem decide se a
etapa deu certo é o artefato em disco, conferido pelo `Trial` — nunca o código de saída.

In [ ]:
class NotebookRunner:
    def __init__(self, logDir, kernel=KERNEL_NAME):
        self.logDir = Path(logDir)
        self.kernel = kernel

    def run(self, path, cwd, replace=None, drop=None, append=None, tolerate=None):
        path = Path(path)
        os.makedirs(self.logDir, exist_ok=True)

        source = self.logDir / f'{path.stem}_in.ipynb'
        out    = self.logDir / f'{path.stem}_out.ipynb'
        nbformat.write(self.build(nbformat.read(path, as_version=4), replace, drop, append), source)

        try:
            pm.execute_notebook(str(source), str(out), kernel_name=self.kernel, cwd=str(cwd), progress_bar=False)
        except pm.PapermillExecutionError as error:
            if not tolerate or tolerate not in str(error):
                raise

        return out

    def build(self, nb, replace=None, drop=None, append=None):
        cells = []

        for cell in nb.cells:
            if drop and cell.cell_type == 'code' and any(key in cell.source for key in drop):
                continue

            if replace and cell.cell_type == 'code':
                for pattern, value in replace.items():
                    cell.source = re.sub(pattern, value, cell.source, flags=re.M)

            cells.append(cell)

        extra = [nbformat.v4.new_code_cell(source) for source in (append or [])]

        for cell in extra if nb.nbformat_minor < 5 else []:
            cell.pop('id', None)   # o Predict.ipynb é 4.4, onde 'id' invalida o notebook e o kernel reclama

        nb.cells = cells + extra
        return nb


preview = NotebookRunner(RUNS_DIR / 'preview').build(
    nbformat.read(MODEL_DIR / 'Predict.ipynb', as_version=4),
    replace={r"^modelId\s*=.*$": "modelId = 'model_99'", r"^dataset\s*=.*$": "dataset = 'dataset_wu'"},
    drop=['plotPrediction(index=i)'])

print(next(c.source for c in preview.cells if 'modelId' in c.source))

# GERAÇÃO DO DATASET SINTÉTICO
O `SyntheticGenerator` roda em **subprocesso** por dois motivos: `Synthetic/index.py` e `Nature/index.py` têm
o mesmo nome de módulo (`index`) e não cabem no mesmo `sys.path`; e gerar 220 volumes de 256³ com 20
processos deixa dezenas de GB de arena de memória para trás, que o subprocesso devolve ao terminar.

`np.random.seed(DATA_SEED)` antes do `dataset()` fixa o `base_seed` que ele sorteia. Sem isso a mesma
configuração geraria dados diferentes a cada chamada, a função objetivo ficaria ruidosa e o DE passaria a
perseguir sorte em vez de sinal.

In [ ]:
GENERATOR = '''
import json, sys
import numpy as np
from index import SyntheticGenerator

options, outputDir, seed, n, jobs = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4]), int(sys.argv[5])
np.random.seed(seed)

gen = SyntheticGenerator()
gen.set(json.load(open(options)))
gen.dataset(n=n, outputDir=outputDir, n_jobs=jobs)
print('gerados', n, 'volumes em', outputDir)
'''


def generate(options, outputDir, path, seed=DATA_SEED, n=N_IMAGES, jobs=N_JOBS):
    Path(path).write_text(json.dumps(options, indent=4))
    args = [sys.executable, '-c', GENERATOR, str(path), str(outputDir), str(seed), str(n), str(jobs)]
    done = subprocess.run(args, cwd=str(SYNTHETIC), capture_output=True, text=True)

    if done.returncode != 0:
        raise RuntimeError(f'geração falhou:\n{done.stderr[-2000:]}')

    return done.stdout.strip().splitlines()[-1]


print(GENERATOR.strip())

# AVALIAÇÃO DE UMA CONFIGURAÇÃO
`Trial` é uma avaliação completa da função objetivo. Cada etapa valida o **artefato** que produziu antes de
passar para a seguinte, porque no papermill um notebook pode "terminar" tendo abortado no meio.

O modelo do trial é identificado por diferença: o conjunto de `Backup/model_*` é fotografado antes do treino
e o que aparecer depois é o desta rodada. Com `KEEP_WEIGHTS = False` o `model.pth` (~600 MB) é apagado logo
depois da medição e o `iou_wu` fica anotado no `info.json` do próprio modelo, junto das `options` que o
geraram — guardar os pesos de 1500 avaliações custaria quase 1 TB que nunca mais seria carregado.

In [ ]:
class Trial:
    def __init__(self, options, tag):
        self.options = options
        self.tag     = tag
        self.logDir  = RUNS_DIR / tag
        self.runner  = NotebookRunner(self.logDir)
        self.model   = None
        self.iou     = None
        self.faults  = None
        self.spread  = None
        self.elapsed = None

    def update(self):
        started = time()
        os.makedirs(self.logDir, exist_ok=True)

        self.generate()
        self.format()
        self.model   = self.train()
        self.iou     = self.predict()
        self.elapsed = time() - started

        self.clean()
        return self.iou

    def generate(self):
        folder = TRIAL_DIR / 'original'
        volumes = Progress('gerando', N_IMAGES, lambda: len(list((folder / 'images').glob('*.npy'))), 'vol')
        volumes.update(lambda: generate(self.options, folder, self.logDir / 'synthetic.json'))

        images = sorted((TRIAL_DIR / 'original' / 'images').glob('*.npy'))
        masks  = sorted((TRIAL_DIR / 'original' / 'masks').glob('*.npy'))

        if len(images) != N_IMAGES or len(masks) != N_IMAGES:
            raise RuntimeError(f'geração produziu {len(images)} imagens e {len(masks)} máscaras, esperado {N_IMAGES}')

        step        = max(1, N_IMAGES // 20)
        self.faults = float(np.mean([np.load(path).mean() for path in masks[::step]]))
        self.spread = float(np.mean([np.load(path).std() for path in images[::step]]))

        # Duas degenerações alcançáveis dentro dos limites que custam um treino inteiro para devolver IoU
        # zero, e que dá para detectar em segundos: dataset sem falha nenhuma (faultCount/faultThrow baixos
        # com faultThreshold alto) e volume sem contraste (dobra extrema satura tudo num valor só; aí o
        # p01 == p99 do Format transforma o dataset inteiro em NaN).
        if self.faults < MIN_FAULT:
            raise RuntimeError(f'dataset com {100 * self.faults:.3f}% de falha (< {100 * MIN_FAULT}%): treino dispensado')

        if self.spread < MIN_SPREAD:
            raise RuntimeError(f'imagens sem contraste (std {self.spread:.4f} < {MIN_SPREAD}): treino dispensado')

    def format(self):
        INFO_PATH.write_text(json.dumps(TRIAL_INFO, indent=4, ensure_ascii=False))
        stamp = time()

        # Conta só o que nasceu nesta rodada: o Format apaga e recria images/, e sem o corte por mtime a
        # barra começaria cheia com os arquivos da avaliação anterior e cairia para zero.
        fresh   = lambda: sum(1 for p in (TRIAL_DIR / 'images').glob('*.npy') if p.stat().st_mtime >= stamp)
        volumes = Progress('formatando', N_IMAGES, fresh, 'vol')
        volumes.update(lambda: self.runner.run(TRIAL_DIR / 'Format.ipynb', cwd=TRIAL_DIR, tolerate='SystemExit'))

        if not DATABASE.exists() or DATABASE.stat().st_mtime < stamp:
            raise RuntimeError('o Format não regravou o DataBase.csv')

        df = pd.read_csv(DATABASE)

        if len(df) != N_IMAGES or not df.img_path.str.contains(f'/{TRIAL_DIR.name}/').all():
            raise RuntimeError(f'DataBase.csv com {len(df)} linhas fora de {TRIAL_DIR.name}')

    def train(self):
        before  = set(os.listdir(BACKUP))
        stamp   = time()
        patch   = {r'^trainer = Trainer\(network, loss, epochs=\d+': f'trainer = Trainer(network, loss, epochs={TRAIN_EPOCHS}'}
        found   = re.search(r'epochs=(\d+)', (MODEL_DIR / 'Analysis.ipynb').read_text())
        epochs  = TRAIN_EPOCHS or (int(found.group(1)) if found else 100)

        # A barra pode terminar antes do total: o early stopping do Model/Analysis.ipynb corta o treino.
        bar = Progress('treinando', epochs, lambda: self.epoch(stamp), 'ép')
        bar.update(lambda: self.runner.run(MODEL_DIR / 'Analysis.ipynb', cwd=MODEL_DIR,
                                           replace=patch if TRAIN_EPOCHS else None))
        created = sorted(set(os.listdir(BACKUP)) - before)

        if len(created) != 1:
            raise RuntimeError(f'esperado 1 modelo novo em Backup/, encontrado {created}')

        if not (BACKUP / created[0] / 'model.pth').exists():
            raise RuntimeError(f'{created[0]} sem model.pth')

        return created[0]

    def epoch(self, since):
        path = MODEL_DIR / 'progress.json'

        if not path.exists() or path.stat().st_mtime < since:
            return 0   # ainda é o progress.json da rodada anterior

        try:
            return json.loads(path.read_text()).get('epoch', 0)
        except (json.JSONDecodeError, OSError):
            return 0   # o Analysis reescreve o arquivo a cada época; a leitura pode cair no meio da escrita

    def predict(self):
        result = self.logDir / 'iou.json'
        record = "{'iou': float(totalIoU), 'model': modelId, 'dataset': dataset, 'samples': len(dfPredict)}"
        append = f'import json\njson.dump({record}, open(r"{result}", "w"), indent=4)'

        # Sem contador: o Predict só escreve no fim, então a barra aqui é cronômetro.
        Progress('predizendo', unit='s').update(
            lambda: self.runner.run(MODEL_DIR / 'Predict.ipynb', cwd=MODEL_DIR, append=[append],
                                    drop=['plotPrediction(index=i)'],
                                    replace={r"^modelId\s*=.*$": f"modelId = '{self.model}'",
                                             r"^dataset\s*=.*$":  f"dataset = '{PREDICT_DATASET}'"}))

        if not result.exists():
            raise RuntimeError('o Predict não gravou o iou.json')

        data = json.loads(result.read_text())

        if data['model'] != self.model or data['dataset'] != PREDICT_DATASET or data['samples'] == 0:
            raise RuntimeError(f'iou.json inconsistente: {data}')

        if not np.isfinite(data['iou']):
            raise RuntimeError(f'IoU não finito: {data}')   # nan viraria penalidade -1e12 e sequestraria o argmax

        return float(data['iou'])

    def clean(self):
        path = BACKUP / self.model / 'info.json'
        data = json.loads(path.read_text())
        data['trainer']['iou_wu'] = self.iou
        data['synthetic'] = self.options
        data['trainer']['faults'] = self.faults
        data['trainer']['spread'] = self.spread
        path.write_text(json.dumps(data, indent=4, ensure_ascii=False))

        if not KEEP_WEIGHTS:
            os.remove(BACKUP / self.model / 'model.pth')

        if not KEEP_LOGS:
            shutil.rmtree(self.logDir, ignore_errors=True)


trial = Trial(BASELINE, 'exemplo')
print(trial.logDir, '\n', json.dumps(trial.options, indent=1)[:200])

# FUNÇÃO OBJETIVO
`Objective` é o que o DE chama. Ela decodifica o genoma, consulta o cache e só então paga o `Trial`.

A chave do cache é o hash das **options do gerador junto do contexto que muda o resultado** (`N_IMAGES`,
`TRIAL_INFO` e o dataset de teste): dois genomas diferentes que arredondam para a mesma configuração produzem
o mesmo dado e o mesmo IoU e a segunda avaliação sai de graça, mas um teste de fumaça com 8 volumes nunca é
confundido com uma avaliação de verdade com 220.

Trial que falha vale `FAILED_SCORE = 0.0`, não penalidade. Um `-1e12` distorceria a escala dos gráficos e
esconderia a diferença entre as configurações boas; um IoU zero diz ao DE exatamente a mesma coisa — aquela
região não serve — e continua legível na análise. A falha fica registrada com a mensagem, e os logs do trial
são preservados em `files/runs/<tag>/` para depuração.

In [ ]:
class Objective:
    def __init__(self, path=TRIALS_PATH):
        self.path   = Path(path)
        self.trials = json.loads(self.path.read_text()) if self.path.exists() else []
        self.cache  = {row['key']: row for row in self.trials}

    def key(self, options):
        context = {'options': options, 'images': N_IMAGES, 'info': TRIAL_INFO, 'dataset': PREDICT_DATASET,
                   'epochs': TRAIN_EPOCHS}
        return hashlib.sha1(json.dumps(context, sort_keys=True).encode()).hexdigest()[:12]

    def __call__(self, values):
        options = toOptions(values)
        key     = self.key(options)

        if key in self.cache:
            return self.cache[key]['iou']

        tag   = f'trial_{len(self.trials) + 1:04d}_{key}'
        row   = {'key': key, 'tag': tag, 'at': datetime.now().isoformat(timespec='seconds'), 'iou': FAILED_SCORE,
                 'model': None, 'images': N_IMAGES, 'faults': None, 'spread': None, 'elapsed': None, 'error': None,
                 'options': options, 'values': values}
        trial = Trial(options, tag)

        try:
            row['iou'] = trial.update()
        except Exception as error:
            row['error'] = f'{type(error).__name__}: {error}'[:500]
            print('FALHOU:', row['error'])

        row['model'], row['faults'], row['spread'], row['elapsed'] = trial.model, trial.faults, trial.spread, trial.elapsed
        self.trials.append(row)
        self.cache[key] = row
        self.path.write_text(json.dumps(self.trials, indent=2, ensure_ascii=False))

        print(f"{tag}  iou={row['iou']:.4f}  modelo={row['model']}  {(row['elapsed'] or 0) / 60:.0f} min")
        return row['iou']

    def df(self):
        if not self.trials:
            return pd.DataFrame(columns=['key', 'tag', 'at', 'iou', 'model', 'images', 'faults', 'spread', 'elapsed', 'error', 'best'])

        df = pd.DataFrame([{k: v for k, v in row.items() if k not in ('options', 'values')} for row in self.trials])
        df['best'] = df.iou.cummax()
        return df


objective = Objective()
print(f'{len(objective.trials)} avaliações em cache')
objective.df().tail()

# TESTE DO PIPELINE
Uma avaliação isolada com o preset atual (`Dataset/dataset_trial/synthetic.json`) antes de largar a campanha.
Serve para duas coisas: provar que as quatro etapas se encaixam nesta máquina e medir o **baseline** — o IoU
que a configuração de hoje alcança, o número que a busca precisa superar.

Para validar só o encanamento, baixe `N_IMAGES` para uns 8 volumes e rode este bloco: o treino termina em
minutos (o early stopping precisa de 16 épocas) e o resultado não polui a campanha, porque `N_IMAGES` entra
na chave do cache. **Volte a 220 antes de otimizar** — e note que o `Trial` sobrescreve
`Dataset/dataset_trial/original/` e o `Dataset/DataBase.csv` a cada chamada.

In [ ]:
RUN_BASELINE = False

if not RUN_BASELINE:
    print('RUN_BASELINE = False — nada executado')

if RUN_BASELINE:
    started  = time()
    baseline = objective(BASELINE_VALUES)
    print(f'baseline iou_wu = {baseline:.4f} em {(time() - started) / 60:.0f} min')

# OTIMIZAÇÃO
`workers=1` é obrigatório: as avaliações compartilham `Task/info.json`, `Dataset/DataBase.csv`,
`Dataset/dataset_trial/` e a GPU — duas em paralelo corromperiam uma à outra. A barra do `AdaptiveDE` conta as
avaliações do ciclo; o log de cada trial sai da própria `Objective`.

Rode esta célula quantas vezes quiser: enquanto o ciclo não fechar, cada chamada continua de onde parou.
Depois que ele fecha (`commit()`), uma chamada nova vira **extensão** — outro ciclo de `GENERATIONS` a partir
dali, com a população reinflada até `POPULATION`.

Uma ressalva sobre `POPULATION = 15` em 37 dimensões: é bem abaixo do default do `AdaptiveDE`
(`18 · nVars = 666`) e menor que a própria dimensão, então os vetores-diferença geram um subespaço de posto
no máximo 14. O crossover binomial ainda mistura coordenadas e evita o travamento completo, mas a exploração
fica limitada. Se conseguir baratear a avaliação com `TRAIN_EPOCHS`/`N_IMAGES`, **aumentar `POPULATION` rende
mais que aumentar `GENERATIONS`** — e a memória aceita a mudança: numa extensão a população é reinflada.

In [ ]:
optimizer = AdaptiveDE(
    objective=objective,
    variables=VARIABLES,
    maximize=True,
    population=POPULATION,
    generations=GENERATIONS,
    variant=VARIANT,
    patience=PATIENCE,
    seed=SEED,
    memory=str(MEMORY_DIR),
    workers=1,
    backend='thread',
    verbose=True
)

pd.Series(optimizer._config())

A célula abaixo é **a campanha** — é ela que gasta horas por avaliação. Construir o `optimizer` (acima) não
roda nada, e é por isso que as duas estão separadas: para analisar os resultados num kernel novo você roda a
célula de cima e **pula esta**, indo direto para a análise.

In [ ]:
best, score = optimizer.update()
print(f'melhor iou_wu = {score:.4f}')
pd.Series(toOptions(best))

# ANÁLISE DOS RESULTADOS
O `plot()` do `Nature` desenha a convergência e o scatter por variável. Ele funciona mesmo num kernel novo,
sem `update()`: a `Memory` reconstrói o `Recorder` a partir do `state.npz`.

In [ ]:
optimizer.plot(save=str(FILES / 'convergence.png'))

In [ ]:
df = objective.df()

plt.figure(figsize=(20, 5))
plt.subplot(1, 2, 1)
plt.plot(df.index, df.iou, 'o', alpha=.5, label='trial')
plt.plot(df.index, df.best, 'r', label='melhor até aqui')
plt.title('IoU no dataset_wu por avaliação'); plt.xlabel('trial'); plt.ylabel('iou_wu')
plt.grid(alpha=.3); plt.legend()

plt.subplot(1, 2, 2)
plt.hist(df.loc[df.iou > 0, 'iou'], bins=30, color='steelblue')
plt.title('Distribuição do IoU dos trials válidos'); plt.xlabel('iou_wu'); plt.ylabel('trials')
plt.grid(alpha=.3)
plt.show()

df.tail(10)

### MELHOR CONFIGURAÇÃO
Exporta o vencedor no mesmo formato do `Dataset/dataset_trial/synthetic.json`, pronto para o `gen.set(...)` do
`Synthetic/Generate.ipynb` ou para virar o preset de um dataset novo.

In [ ]:
memory = json.loads((MEMORY_DIR / 'best.json').read_text())
winner = toOptions(memory['best'])
(FILES / 'synthetic.json').write_text(json.dumps(winner, indent=4))

print(f"melhor de {memory['runs']} campanha(s): iou_wu = {memory['score']:.4f}")
pd.DataFrame({'baseline': BASELINE, 'otimizado': winner})

In [ ]:
history = pd.json_normalize(json.loads((MEMORY_DIR / 'history.json').read_text()))
history[['run', 'at', 'model', 'score', 'improved', 'stopped']]